In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "farhan@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    Authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for the recent emails"""

    return """
    Hey Farhan I am in the town for a week, How about we meet and discuss business,
    if you can make some time and come for a dinner let me know,
    -best, Julie(julie@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response Email"""

    return f"Email Sent to {to} with subject {subject} body {body}"


@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the User with given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(
            update={
                "Authenticated" : True,
                "messages" : [ToolMessage("Successfully authenticated", tool_call_id=runtime.tool_call_id)],
            }
        )

    else:
        return Command(
            update={
                "Authenticated" : False,
                "messages" : [ToolMessage("Authentication failed", tool_call_id=runtime.tool_call_id)]
            }
        )

In [5]:
from langchain.agents.middleware import wrap_model_call,ModelRequest,ModelResponse
from typing import Callable

@wrap_model_call
async def dynamic_tool_call(
        request: ModelRequest, handler: Callable[[ModelRequest],ModelResponse]
) -> ModelResponse:
    """Allow read inbox and send Email tools only if user provide correct email and password"""

    authenticated = request.state.get("Authenticated")

    if authenticated:
        tools=[check_inbox,send_email]
    else:
        tools=[authenticate]

    request=request.override(tools=tools)
    return await handler(request)


In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails.
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."


@dynamic_prompt
def custom_auth_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("Authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        custom_auth_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
    ]
)


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Please authenticate my account using farhan@example.com and password123")]},
    context=EmailContext(),
    config=config
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'You have an email from Julie (julie@example.com) regarding a business discussion over dinner. Would you like to respond to her?', 'extras': {'signature': 'Ct0BAWkUfROlFNb0pvZXeDbxe2hQ0a8nEFVrnR7MZbbKCxIgQq4sO9gKNBuxt7HBW/V9vo12vmCJ17/6IPDURq3wWmKRZ5/Lt0mthZiI27smXc+v7fOednOl9cnpjZeVeCejyBffJ3NejeRJF/fA6bQcSCnoiQhfOemufxwlWTvziGcpr1y8S79r41kHrf4J+2uaHs5wgxCnP2DdkyJkmxB+lF5iuRfltexJ2Yia2DBdseJUMG1QTH7d+u4dnz9rpojhd5yznBR28EQZoaKjY+yHMh+5ou3qZg6ZGLuWZb4='}}]


In [12]:

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Check my inbox. Then, draft a reply to the email. that i will meet")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

[{'type': 'text', 'text': 'I have already checked your inbox. You have an email from Julie (julie@example.com) regarding a business discussion over dinner.\n\nYou mentioned you want to meet. What should be the subject of the email, and what specific message would you like to convey to Julie? Please include details like your availability or any preferences for the meeting.', 'extras': {'signature': 'CqACAWkUfRMYvuiPmD24qlmI7aelKA5JikJwAQsuv2GhZhjKt18e8RBo9rElftJ8PGl2+6o68NFI9OP6lVjHh7fwkfxcGZzOQZYjqLkMB6OQ/NoP7KF6B5paG+GGri2ui2RSJs0H1mBbO6Is84QGuvM33Rl7EeaO1eq6iF8BVPWtJCopvc1iB9tM9mJjEBeNk/AIsRQXjmwwRWcEls/E+wmu6db6iTH5iWlv8CVgDZLFUt/fRICW0CHlmDMW/t6p7jvoLsmJBL33Qg9JIF+RhlWgSN3COV1a8irRH9OpZqKusvdG1apJBqzqFrL5IDLzehzC34l0rwn2jAD8efQEJ9sq2EDFkJ+vIl6hwAPJjE/MiUXj0ShamxuMbL8MzwODDiOq'}}]


In [13]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

In [14]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I have already checked your inbox. You have an email from Julie (julie@example.com) regarding a business discussion over dinner.\n\nYou mentioned you want to meet. What should be the subject of the email, and what specific message would you like to convey to Julie? Please include details like your availability or any preferences for the meeting.', 'extras': {'signature': 'CqACAWkUfRMYvuiPmD24qlmI7aelKA5JikJwAQsuv2GhZhjKt18e8RBo9rElftJ8PGl2+6o68NFI9OP6lVjHh7fwkfxcGZzOQZYjqLkMB6OQ/NoP7KF6B5paG+GGri2ui2RSJs0H1mBbO6Is84QGuvM33Rl7EeaO1eq6iF8BVPWtJCopvc1iB9tM9mJjEBeNk/AIsRQXjmwwRWcEls/E+wmu6db6iTH5iWlv8CVgDZLFUt/fRICW0CHlmDMW/t6p7jvoLsmJBL33Qg9JIF+RhlWgSN3COV1a8irRH9OpZqKusvdG1apJBqzqFrL5IDLzehzC34l0rwn2jAD8efQEJ9sq2EDFkJ+vIl6hwAPJjE/MiUXj0ShamxuMbL8MzwODDiOq'}}]


In [15]:
from pprint import pprint

pprint(response)

{'Authenticated': True,
 'messages': [HumanMessage(content='Please authenticate my account using farhan@example.com and password123', additional_kwargs={}, response_metadata={}, id='6b94436f-cbe5-4fa5-ae87-dd6ba3f2447b'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'authenticate', 'arguments': '{"email": "farhan@example.com", "password": "password123"}'}, '__gemini_function_call_thought_signatures__': {'66d4338e-5813-4697-89ec-d6d07baf7686': 'CvoCAWkUfRMTo0vWcBi5qrQZqcfRyUotUUQImkU72gjF/zrCRk7a0id1NBriNJBlMpBMSFgxax3KMOTLwYlqfgRUbdge9ff2N1cfAqR5Uj/h0oBMBnvHUKnr3LybZH8xFZzNRPQDMJ6RcvKGItB3p+/zXmU4BDs7oRKnwdFrtEeCm3Z51onQ9X+GSIsY3cDNx6Px7R9K0cYPyO+OgeVdgYjr+RkKo8DKLL1Ohe27IFAbASV/t3+6fTDMCQH6pjYml4CgF5v6SrWW0XaXVxmHLEzCQOJKMEx5pzxxhDeu9nQKZou+zfDQ08CFk7143yiPX77Yo7V2zjy1uHbLhMBOJa8MeZWvXJ3gL72Ki/4P0s88OSvgZ+vgcWFO+PcL8ANa6Tqho9s5L1T00xGt/KF8xIU9KZgne807y7xALS8LIQxnfq5JWn1tuK1nXVcs2hRdq/wkA4vQlbenvyL113EgFqSe8HG4qNS0YP3A76tKMztVKwrDMtaxZyd5vOrh'}}, res